# Jump Stein Variational Gradient Descent (J-SVGD) 

In [ ]:
# JAX >= 0.4
from typing import Callable, Dict, Tuple
import jax
import jax.numpy as jnp
from jax import random, vmap, jit

Array = jnp.ndarray
ProposeFn = Callable[..., Tuple[Array, Array, Array]]   # returns (y, logq_xy, logq_yx)
LogProbFn = Callable[[Array], Array]                    # (N,D)->(N,)

# ------------------------------------------------------------
# Kernels
# ------------------------------------------------------------

def rbf_kernel(X: Array, Z: Array, lengthscale: float = 1.0) -> Array:
    """
    RBF kernel K(X,Z) with isotropic lengthscale.
    X: (N,D), Z: (K,D) -> Kxz: (N,K)
    """
    X2 = jnp.sum(X**2, axis=-1, keepdims=True)          # (N,1)
    Z2 = jnp.sum(Z**2, axis=-1, keepdims=True).T        # (1,K)
    sqdist = X2 + Z2 - 2.0 * (X @ Z.T)                  # (N,K)
    return jnp.exp(-0.5 * sqdist / (lengthscale**2))

# You can add other kernels here (e.g. linear, Matérn) with same signature.

# ------------------------------------------------------------
# Build witness psi*(x) at current particles X
# ------------------------------------------------------------

@jit
def jsvgd_witness(
    key: random.PRNGKey,
    X: Array,                          # (N,D)
    logpi_fn: LogProbFn,
    propose_fn: ProposeFn,
    mh_accept_prob: Callable[[Array, Array, Array, Array], Array],
    M: int,
    kernel_fn: Callable[[Array, Array], Array] = rbf_kernel,
    kernel_kwargs: Dict = None,
    propose_kwargs: Dict = None,
    ridge_lambda: float = 0.0,
    normalise: bool = True,
) -> Array:
    """
    Compute psi_t^*(X) for J-SVGD:
        h_hat(·) = (1/(N M)) sum_{i,m} alpha_{i,m} [k(·, y_{i,m}) - k(·, x_i)]
    and return psi^*(X) = h_hat evaluated at X (shape (N,)).
    For vector-valued psi, you typically apply one scalar witness per coordinate
    by using a product kernel or separate kernels per dim; here we use a scalar RKHS
    that produces a scalar field psi evaluated at particle locations (standard SVGD-style).
    """
    if kernel_kwargs is None:
        kernel_kwargs = {}
    if propose_kwargs is None:
        propose_kwargs = {}

    N, D = X.shape
    # Draw M proposals per particle
    # keys_im: (N,M,2) -> we’ll consume one per (i,m) inside propose_fn via split
    keys = random.split(key, N * M)
    keys = keys.reshape(N, M, 2)  # keep extra split room if propose_fn needs multiple

    # Compute logπ(x) once
    logpi_x = logpi_fn(X)  # (N,)

    # vmap over m, then over i
    def propose_one_for_particle(x_i, key_row):
        """
        Returns:
          Y_i: (M,D), logq_xy_i: (M,), logq_yx_i: (M,), logpi_y_i: (M,)
        """
        def per_m(km):
            k_main = km[0]  # use first subkey
            y, logq_xy, logq_yx = propose_fn(k_main, x_i, **propose_kwargs)
            return y, logq_xy, logq_yx

        Y_i, logq_xy_i, logq_yx_i = jax.vmap(per_m)(key_row)  # (M,D), (M,), (M,)
        logpi_y_i = logpi_fn(Y_i)                              # (M,)
        return Y_i, logq_xy_i, logq_yx_i, logpi_y_i

    Y, logq_xy, logq_yx, logpi_y = jax.vmap(propose_one_for_particle, in_axes=(0,0))(X, keys)
    # Shapes:
    #   Y:         (N,M,D)
    #   logq_xy:   (N,M)
    #   logq_yx:   (N,M)
    #   logpi_y:   (N,M)

    # MH acceptance α_{i,m}
    # broadcast logpi_x to (N,M)
    alpha = mh_accept_prob(
        logpi_x[:, None],  # (N,1)
        logpi_y,           # (N,M)
        logq_xy,           # (N,M)
        logq_yx            # (N,M)
    )  # -> (N,M)

    # Flatten Y to (NM, D) and weights to (NM,)
    Y_flat = Y.reshape(N * M, D)
    w_im = (alpha / (N * M)).reshape(N * M)         # weights for k(·, y_{i,m})
    # Sum over m for each i: (N,)
    a_i_sum = (alpha.sum(axis=1) / (N * M))         # weights for k(·, x_i)

    # Compute kernel matrices:
    # K_XY: (N, NM) with entries k(x_j, y_{i,m})
    K_XY = kernel_fn(X, Y_flat, **kernel_kwargs)
    # K_XX: (N, N) with entries k(x_j, x_i)
    K_XX = kernel_fn(X, X, **kernel_kwargs)

    # h_hat evaluated at X: (N,)
    #   h(X) = K_XY @ w_im  -  K_XX @ a_i_sum
    h_vals = (K_XY @ w_im) - (K_XX @ a_i_sum)

    # Optional stabilising normalisation (empirical)
    if normalise:
        scale = jnp.sqrt(jnp.mean(h_vals**2) + ridge_lambda)
        h_vals = h_vals / jnp.where(scale > 0.0, scale, 1.0)

    # psi^*(X) = h_hat(X)
    return h_vals  # (N,)

# ------------------------------------------------------------
# One J-SVGD step: X <- X + eps * psi^*(X)
# ------------------------------------------------------------

@jit
def jsvgd_step(
    key: Array,
    X: Array,                          # (N,D)
    logpi_fn: LogProbFn,
    propose_fn: ProposeFn,
    mh_accept_prob: Callable[[Array, Array, Array, Array], Array],
    eps: float,
    M: int,
    kernel_fn: Callable[[Array, Array], Array] = rbf_kernel,
    kernel_kwargs: Dict = None,
    propose_kwargs: Dict = None,
    ridge_lambda: float = 0.0,
    normalise: bool = True,
) -> Array:
    """
    Perform one J-SVGD update: X_next = X + eps * psi^*(X),
    where psi^*(X) is the scalar witness evaluated at particle locations.
    For vector-valued transports, apply per-coordinate or with product kernels.
    """
    psi_vals = jsvgd_witness(
        key=key,
        X=X,
        logpi_fn=logpi_fn,
        propose_fn=propose_fn,
        mh_accept_prob=mh_accept_prob,
        M=M,
        kernel_fn=kernel_fn,
        kernel_kwargs=kernel_kwargs,
        propose_kwargs=propose_kwargs,
        ridge_lambda=ridge_lambda,
        normalise=normalise,
    )  # (N,)

    # Lift scalar field to a vector update direction.
    # Simplest choice: multiply by a unit direction per particle (e.g., score or random).
    # Here we push along the per-particle score direction for a meaningful vector field:
    score = jax.grad(lambda Z: logpi_fn(Z).sum())(X)  # (N,D)
    # Normalise score to avoid exploding steps
    score_norm = jnp.linalg.norm(score, axis=-1, keepdims=True)
    score_unit = score / (score_norm + 1e-8)

    X_next = X + eps * psi_vals[:, None] * score_unit
    return X_next
